In [ ]:
from itertools import combinations
from math import isfinite

import pandas as pd


# ============================================================
# Configuration
# ============================================================

CSV_PATH = "data/annotation/face_acts_annotators.csv"

ANNOT_COLS = [
    "FaceActsAdmin",
    "FaceActsBrianna",
    "FaceActsAmber",
]


# ============================================================
# Data loading
# ============================================================

def load_csv_robust(path):
    """
    Load a CSV file, supporting either comma- or semicolon-separated files.
    """
    try:
        df = pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, sep=";")

    if df.shape[1] == 1 or not set(ANNOT_COLS).issubset(df.columns):
        df = pd.read_csv(path, sep=";")

    return df


# ============================================================
# Annotation parsing and distance
# ============================================================

def parse_set(cell):
    """
    Convert a multi-label annotation into a frozenset.

    Example:
        "HPos-, HNeg-" -> frozenset({"HPos-", "HNeg-"})

    Empty or missing annotations return None.
    """
    if pd.isna(cell):
        return None

    labels = [label.strip() for label in str(cell).split(",")]
    labels = {label for label in labels if label}

    return frozenset(labels) if labels else None


def jaccard_distance(a, b):
    """
    Compute Jaccard set distance:

        1 - |A ∩ B| / |A ∪ B|
    """
    intersection = len(a & b)
    union = len(a | b)

    return 1.0 - (intersection / union) if union else 0.0


# ============================================================
# Krippendorff's alpha
# ============================================================

def krippendorff_alpha_set_distance(
    df,
    annot_cols,
    distance_fn=jaccard_distance,
):
    """
    Compute Krippendorff's alpha for multi-label annotations
    using Jaccard set distance.
    """

    annotations_by_item = []
    pooled_annotations = []

    # Keep items with at least two valid annotators.
    for _, row in df[annot_cols].iterrows():
        values = [parse_set(row[col]) for col in annot_cols]
        values = [value for value in values if value is not None]

        if len(values) >= 2:
            annotations_by_item.append(values)
            pooled_annotations.extend(values)

    # --------------------------------------------------------
    # Observed disagreement
    # --------------------------------------------------------

    observed_pairs = sum(
        len(values) * (len(values) - 1) // 2
        for values in annotations_by_item
    )

    if observed_pairs == 0:
        return (
            float("nan"),
            float("nan"),
            float("nan"),
            0.0,
            0.0,
        )

    Do = sum(
        distance_fn(a, b)
        for values in annotations_by_item
        for a, b in combinations(values, 2)
    ) / observed_pairs

    # --------------------------------------------------------
    # Expected disagreement
    # --------------------------------------------------------

    expected_pairs = (
        len(pooled_annotations)
        * (len(pooled_annotations) - 1)
        // 2
    )

    if expected_pairs == 0:
        De = 0.0
        alpha = 1.0 if Do == 0 else 0.0

        return (
            alpha,
            Do,
            De,
            0.0,
            0.0,
        )

    De = sum(
        distance_fn(a, b)
        for a, b in combinations(pooled_annotations, 2)
    ) / expected_pairs

    # --------------------------------------------------------
    # Krippendorff's alpha
    # --------------------------------------------------------

    if De == 0:
        alpha = 1.0 if Do == 0 else 0.0
    else:
        alpha = 1.0 - (Do / De)

    # --------------------------------------------------------
    # Additional descriptive statistics
    # --------------------------------------------------------

    items_used = len(annotations_by_item)

    multi_label_items = sum(
        any(len(annotation_set) > 1 for annotation_set in values)
        for values in annotations_by_item
    )

    multi_label_share = (
        multi_label_items / items_used
        if items_used
        else 0.0
    )

    exact_consensus_rate = (
        sum(
            all(values[0] == other for other in values)
            for values in annotations_by_item
        )
        / items_used
        if items_used
        else 0.0
    )

    return (
        alpha,
        Do,
        De,
        multi_label_share,
        exact_consensus_rate,
    )


# ============================================================
# Run
# ============================================================

if __name__ == "__main__":

    df = load_csv_robust(CSV_PATH)

    missing_columns = [
        col for col in ANNOT_COLS
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing annotator columns: {missing_columns}. "
            f"Found columns: {list(df.columns)}"
        )

    (
        alpha,
        Do,
        De,
        multi_label_share,
        exact_consensus,
    ) = krippendorff_alpha_set_distance(
        df,
        ANNOT_COLS,
        jaccard_distance,
    )

    print("========== Results: Jaccard Set Distance ==========")

    print(
        f"Share with any multi-label:    "
        f"{multi_label_share * 100:.2f}%"
    )

    print(
        f"Exact set consensus rate:      "
        f"{exact_consensus * 100:.2f}%"
    )

    print(f"Observed disagreement (Do):    {Do:.6f}")
    print(f"Expected disagreement (De):    {De:.6f}")

    if isfinite(alpha):
        print(
            f"Krippendorff's alpha "
            f"(Jaccard):              {alpha:.6f}"
        )
    else:
        print(
            "Krippendorff's alpha "
            "(Jaccard):              NaN "
            "(insufficient data)"
        )